In [43]:
import subprocess, os, json, requests, re, time, pynvml, threading
from evalplus.data import get_human_eval_plus, get_mbpp_plus

In [44]:
path = "../data/processed/selfdebug/qwen_3b"
humaneval_filename = "humaneval_solutions.jsonl"
mbpp_filename = "mbpp_solutions.jsonl"
ollama_url = "http://host.docker.internal:11434/api/generate"
max_iterations = 3

In [45]:
def test_solution(solution_code, test_code, entry_point, timeout=5):
    script = f"{solution_code}\n\n{test_code}"
    if "def check(" in test_code:
        script = f"{script}\n\ncheck({entry_point})\n\n"

    try:
        result = subprocess.run(
            ["python3", "-c", script],
            capture_output=True,
            text=True,
            timeout=timeout
        )

        if result.returncode == 0:
            return True, ""
        else:
            return False, result.stderr.strip()

    except subprocess.TimeoutExpired:
        return False, "TimeoutExpired: the code entered an infinite loop, or spent too much time."
    except Exception as e:
        return False, f"Execution Error: {str(e)}"

    

In [46]:
def clean_code(code):
    regex = r"```python\n(.*?)\n```"
    match = re.search(regex, code, re.DOTALL)
    return match.group(1) if match else code

In [47]:
def monitor_GPU(handle):
	global energy, peak_vram, monitoring
	power_supported = True
	last_time = time.time()
	while monitoring:
		now = time.time()
		time_diff = now-last_time
		last_time = now

		if power_supported:
			try:
				watt = pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0 # function returns mW
				energy += watt * time_diff
			except pynvml.NVMLError_NotSupported:
				power_supported = False

		memory = pynvml.nvmlDeviceGetMemoryInfo(handle)
		vram = memory.used / (1024 ** 2)
		peak_vram = vram if vram > peak_vram else peak_vram

		time.sleep(0.1)

In [48]:
def generate_solution(problems, output_file):
    global energy, peak_vram, monitoring

    completed_tasks = set()
    if not os.path.exists(path):
        os.makedirs(path)

    if os.path.exists(output_file):
        with open(output_file, 'r') as f:
            for line in f:
                completed_tasks.add(json.loads(line.strip())["task_id"])

    print(f"loaded {len(completed_tasks)} solutions.\n")
    if(len(problems) - len(completed_tasks) == 0):
        print("solutions already generated.\n")
        return

    energy = 0.0
    peak_vram = 0.0
    monitoring = True

    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)

    with open(output_file, 'a') as f:
        thread = threading.Thread(target=monitor_GPU, args=(handle,))
        thread.start()
        start_time = time.time()

        for task_id in problems.keys():
            if task_id in completed_tasks: continue
            problem = problems[task_id]
            prompt = problem["prompt"]

            i = 0
            while True:
                payload = {
                    "model": "qwen2.5-coder:3b",
                    "prompt": prompt,
                    "stream": False
                }
                response = requests.post(ollama_url, json=payload).json()
                generated_code = clean_code(response.get("response", ""))

                test_code = problem["test"] if "test" in problem.keys() else problem["assertion"]
                test_result, test_error = test_solution(generated_code, test_code, problem["entry_point"])
                if test_result or i > max_iterations:
                    break

                i += 1
                print(f"problem {task_id} has failed the test")
                prompt = f"The following code is not correct: {generated_code}\n\nDuring execution the following exception has been encountered: {test_error}\n\nRewrite the function fixing the error."

            f.write(json.dumps({
                "task_id": task_id,
                "solution": generated_code
            }) + "\n")

        monitoring = False
        thread.join()
        total_time = time.time()-start_time
        print(f"time: {total_time}\nenergy: {energy}\npeak_vram: {peak_vram}\n")
        pynvml.nvmlShutdown()


In [49]:
humaneval_problems = get_human_eval_plus()
generate_solution(humaneval_problems, f"{path}/{humaneval_filename}")

loaded 0 solutions.

problem HumanEval/8 has failed the test
problem HumanEval/20 has failed the test
problem HumanEval/25 has failed the test
problem HumanEval/26 has failed the test
problem HumanEval/26 has failed the test
problem HumanEval/26 has failed the test
problem HumanEval/26 has failed the test
problem HumanEval/38 has failed the test
problem HumanEval/38 has failed the test
problem HumanEval/38 has failed the test
problem HumanEval/38 has failed the test
problem HumanEval/46 has failed the test
problem HumanEval/46 has failed the test
problem HumanEval/46 has failed the test
problem HumanEval/46 has failed the test
problem HumanEval/50 has failed the test
problem HumanEval/50 has failed the test
problem HumanEval/50 has failed the test
problem HumanEval/50 has failed the test
problem HumanEval/62 has failed the test
problem HumanEval/67 has failed the test
problem HumanEval/67 has failed the test
problem HumanEval/67 has failed the test
problem HumanEval/74 has failed the t

In [50]:
mbpp_problems = get_mbpp_plus()
generate_solution(mbpp_problems, f"{path}/{mbpp_filename}")

loaded 0 solutions.

problem Mbpp/2 has failed the test
problem Mbpp/2 has failed the test
problem Mbpp/2 has failed the test
problem Mbpp/2 has failed the test
problem Mbpp/4 has failed the test
problem Mbpp/9 has failed the test
problem Mbpp/9 has failed the test
problem Mbpp/11 has failed the test
problem Mbpp/20 has failed the test
problem Mbpp/20 has failed the test
problem Mbpp/20 has failed the test
problem Mbpp/20 has failed the test
problem Mbpp/61 has failed the test
problem Mbpp/61 has failed the test
problem Mbpp/61 has failed the test
problem Mbpp/61 has failed the test
problem Mbpp/63 has failed the test
problem Mbpp/63 has failed the test
problem Mbpp/63 has failed the test
problem Mbpp/63 has failed the test
problem Mbpp/67 has failed the test
problem Mbpp/69 has failed the test
problem Mbpp/69 has failed the test
problem Mbpp/69 has failed the test
problem Mbpp/69 has failed the test
problem Mbpp/70 has failed the test
problem Mbpp/70 has failed the test
problem Mbpp/7

In [51]:
!evalplus.evaluate --dataset humaneval --samples {path}/{humaneval_filename}
!evalplus.evaluate --dataset mbpp --samples {path}/{mbpp_filename}

Computing expected output...
Expected outputs computed in 15.99s
Reading samples...
164it [00:00, 1604.95it/s]
100%|████████████████████████████████████████| 164/164 [00:10<00:00, 15.72it/s]
humaneval (base tests)
pass@1:	0.841
humaneval+ (base + extra tests)
pass@1:	0.793
Load from ground-truth from /home/vscode/.cache/evalplus/ee43ecabebf20deef4bb776a405ac5b1.pkl
Reading samples...
378it [00:00, 1650.17it/s]
100%|████████████████████████████████████████| 378/378 [00:46<00:00,  8.21it/s]
mbpp (base tests)
pass@1:	0.770
mbpp+ (base + extra tests)
pass@1:	0.638
